Same question as the pathway-coverage report, but for traits: the query set is phenotypes (PhenomeXcan rapid GWAS, scored via phenoplier's GLS) rather than gene sets, and the denominator is the fixed trait catalog size. Each model is fit at its native full rank (no forced K) across study-subsampling fractions 1-75%; the 100% cell is the same full-compendium fit already shown in the pathway-coverage report, so it's dropped here. CLAMPfull_bp and CLAMPbase are reported as separate panels, one colour each with an unpaired, one-sided Student's t-test (pooled variance) against 75%, matching the pathway-coverage report's style.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
suppressPackageStartupMessages({
    library(data.table)
    library(ggplot2)
    library(ggpubr)
    library(here)
})

traits <- fread(here(snakemake@input[["traits_long"]]))
FDR <- unique(traits$fdr)
stopifnot(length(FDR) == 1)

traits[, model := factor(model, levels = c("CLAMPfull_bp", "CLAMPbase"))]
traits <- traits[fraction < 100]
traits[, fraction_label := factor(sprintf("%d%%", fraction),
                                   levels = sprintf("%d%%", sort(unique(fraction))))]
traits[, .N, by = .(model, fraction_label)]

## Trait recovery by sample fraction

In [ ]:
base_colors <- unlist(snakemake@config[["archs4"]][["coverage"]][["colors"]][["model"]])
model_colors <- c(
    CLAMPfull_bp = unname(base_colors[["CLAMPfull"]]),
    CLAMPbase = unname(base_colors[["CLAMPbase"]])
)

fraction_totals <- traits[, .(
    recovered_traits = as.integer(round(median(recovered_traits))),
    eligible_traits = as.integer(round(median(eligible_traits)))
), by = .(fraction_label, model)]
fraction_totals[, label := sprintf(
    "%s (%.0f%%)", format(recovered_traits, big.mark = ",", trim = TRUE),
    100 * recovered_traits / eligible_traits
)]

REF_FRACTION <- "75%"
COMPARE_FRACTIONS <- c("25%", "50%")

make_fraction_plot <- function(model_name) {
    d <- traits[model == model_name]
    totals <- fraction_totals[model == model_name]
    stopifnot(all(d[, .N, by = fraction_label]$N == 3L))
    setorder(d, fraction_label, seed)
    comparisons <- rbindlist(lapply(COMPARE_FRACTIONS, function(fl) {
        reference <- d[fraction_label == REF_FRACTION][order(seed)]$recovered_traits
        candidate <- d[fraction_label == fl][order(seed)]$recovered_traits
        data.table(
            group1 = fl, group2 = REF_FRACTION,
            p = t.test(reference, candidate, paired = FALSE, alternative = "greater", var.equal = TRUE)$p.value,
            difference = as.integer(round(mean(reference) - mean(candidate)))
        )
    }))
    comparisons[, y.position := max(d$recovered_traits) + seq_len(.N) * diff(range(d$recovered_traits)) * 0.06]
    comparisons[, label := sprintf("p = %.3f; %+d traits", p, difference)]
    totals[, label_y := recovered_traits - 0.10 * diff(range(d$recovered_traits))]

    print(ggplot(d, aes(fraction_label, recovered_traits)) +
        geom_boxplot(fill = model_colors[[model_name]], width = 0.65, outlier.shape = NA) +
        stat_summary(aes(group = 1), fun = median, geom = "line", linetype = "dashed",
                     colour = "#222222", linewidth = 0.6) +
        geom_jitter(width = 0.1, height = 0, size = 2.5) +
        geom_text(data = totals, aes(fraction_label, label_y, label = label), inherit.aes = FALSE, size = 3.2) +
        ggpubr::stat_pvalue_manual(comparisons, label = "label", tip.length = 0.01, size = 3) +
        scale_y_continuous(expand = expansion(mult = c(0.14, 0.18))) +
        labs(
            x = "Training compendium coverage",
            y = sprintf("Recovered traits (of %d tested)", unique(d$eligible_traits)),
            title = sprintf("%s - trait recovery by sample fraction (FDR %.2f)", model_name, FDR)
        ) +
        theme_classic(base_size = 15) +
        theme(plot.title = element_text(face = "bold", size = 14)))
}

options(repr.plot.width = 11, repr.plot.height = 6.5)
for (m in levels(traits$model)) make_fraction_plot(m)

## Trait recovery at full data (cross-compendium)

In [ ]:
finals <- fread(here(snakemake@input[["finals_long"]]))
finals[, model := factor(model, levels = c("CLAMPfull_bp", "CLAMPbase"))]

dataset_names <- c(archs4 = "ARCHS4", gtex = "GTEx", recount2 = "recount2")
dataset_colors <- unlist(snakemake@config[["archs4"]][["coverage"]][["colors"]][["dataset"]])
finals[, label := format(recovered_traits, big.mark = ",", trim = TRUE)]

make_cross_plot <- function(model_name) {
    d <- finals[model == model_name]
    d[, dataset_label := factor(dataset, levels = names(dataset_names), labels = dataset_names)]
    print(ggplot(d, aes(dataset_label, recovered_traits, colour = dataset_label)) +
        geom_point(size = 3.5) +
        geom_text(aes(label = label), vjust = -0.8, colour = "black", size = 3.5) +
        scale_colour_manual(values = dataset_colors, guide = "none") +
        scale_y_continuous(expand = expansion(mult = c(0.08, 0.16))) +
        labs(
            x = NULL,
            y = sprintf("Recovered traits (of %d tested)", unique(d$eligible_traits)),
            title = sprintf("%s - full-data comparison across compendia (FDR %.2f)", model_name, FDR)
        ) +
        theme_classic(base_size = 15) +
        theme(plot.title = element_text(face = "bold", size = 14)))
}

options(repr.plot.width = 8, repr.plot.height = 6)
for (m in levels(finals$model)) make_cross_plot(m)